In [1]:
import polars as pl
from pathlib import Path
from glob import glob

DATA = Path("../data")
print("Polars version:", pl.__version__)
print("Data folder exists:", DATA.exists())

Polars version: 1.38.1
Data folder exists: True


In [2]:
labels   = pl.read_parquet(DATA / "train_labels.parquet")
accounts = pl.read_parquet(DATA / "accounts.parquet")
customers = pl.read_parquet(DATA / "customers.parquet")
linkage  = pl.read_parquet(DATA / "customer_account_linkage.parquet")
products = pl.read_parquet(DATA / "product_details.parquet")
demo     = pl.read_parquet(DATA / "demographics.parquet")
branch   = pl.read_parquet(DATA / "branch.parquet")
acct_add = pl.read_parquet(DATA / "accounts-additional.parquet")
test     = pl.read_parquet(DATA / "test_accounts.parquet")

for name, df in [
    ("labels", labels), ("accounts", accounts), ("customers", customers),
    ("linkage", linkage), ("products", products), ("demographics", demo),
    ("branch", branch), ("acct_additional", acct_add), ("test", test)
]:
    print(f"{name:25s}: {df.shape}")

labels                   : (96091, 5)
accounts                 : (160153, 22)
customers                : (159416, 14)
linkage                  : (160153, 2)
products                 : (159416, 11)
demographics             : (159416, 9)
branch                   : (9000, 9)
acct_additional          : (160153, 2)
test                     : (64062, 1)


In [3]:
print(labels["is_mule"].value_counts())
print(f"\nMule rate: {labels['is_mule'].mean():.4%}")
print(f"Total mules: {labels['is_mule'].sum()}")
print(f"Total legit: {(labels['is_mule'] == 0).sum()}")

shape: (2, 2)
┌─────────┬───────┐
│ is_mule ┆ count │
│ ---     ┆ ---   │
│ i64     ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 93408 │
│ 1       ┆ 2683  │
└─────────┴───────┘

Mule rate: 2.7921%
Total mules: 2683
Total legit: 93408


In [4]:
mules = labels.filter(pl.col("is_mule") == 1)
legit = labels.filter(pl.col("is_mule") == 0)

print("=== Mule columns ===")
print(mules.head(3))
print("\n=== Legit columns ===")
print(legit.head(3))

print("\n=== Null counts — Mules ===")
print(mules.null_count())
print("\n=== Null counts — Legit ===")
print(legit.null_count())

=== Mule columns ===
shape: (3, 5)
┌─────────────┬─────────┬────────────────┬─────────────────────────────┬───────────────────┐
│ account_id  ┆ is_mule ┆ mule_flag_date ┆ alert_reason                ┆ flagged_by_branch │
│ ---         ┆ ---     ┆ ---            ┆ ---                         ┆ ---               │
│ str         ┆ i64     ┆ str            ┆ str                         ┆ f64               │
╞═════════════╪═════════╪════════════════╪═════════════════════════════╪═══════════════════╡
│ ACCT_000038 ┆ 1       ┆ 2022-08-19     ┆ Income-Transaction Mismatch ┆ 7361.0            │
│ ACCT_000087 ┆ 1       ┆ 2021-02-17     ┆ Routine Investigation       ┆ 1389.0            │
│ ACCT_000135 ┆ 1       ┆ 2025-05-31     ┆ Layered Transaction Pattern ┆ 6200.0            │
└─────────────┴─────────┴────────────────┴─────────────────────────────┴───────────────────┘

=== Legit columns ===
shape: (3, 5)
┌─────────────┬─────────┬────────────────┬──────────────┬───────────────────┐
│ account_id 

In [5]:
# Save these suspicious mules now — no alert reason despite being labelled mule
suspicious_mules = mules.filter(pl.col("alert_reason").is_null())
print(f"Mules with no alert reason: {len(suspicious_mules)}")
print(suspicious_mules.head(10))

Mules with no alert reason: 245
shape: (10, 5)
┌─────────────┬─────────┬────────────────┬──────────────┬───────────────────┐
│ account_id  ┆ is_mule ┆ mule_flag_date ┆ alert_reason ┆ flagged_by_branch │
│ ---         ┆ ---     ┆ ---            ┆ ---          ┆ ---               │
│ str         ┆ i64     ┆ str            ┆ str          ┆ f64               │
╞═════════════╪═════════╪════════════════╪══════════════╪═══════════════════╡
│ ACCT_000783 ┆ 1       ┆ 2022-03-31     ┆ null         ┆ 5091.0            │
│ ACCT_001982 ┆ 1       ┆ 2025-04-29     ┆ null         ┆ 9240.0            │
│ ACCT_002610 ┆ 1       ┆ 2025-11-29     ┆ null         ┆ 4292.0            │
│ ACCT_003277 ┆ 1       ┆ 2025-05-26     ┆ null         ┆ 2050.0            │
│ ACCT_004106 ┆ 1       ┆ 2024-04-10     ┆ null         ┆ 4462.0            │
│ ACCT_004470 ┆ 1       ┆ 2025-04-15     ┆ null         ┆ 2681.0            │
│ ACCT_005083 ┆ 1       ┆ 2025-09-16     ┆ null         ┆ 1011.0            │
│ ACCT_007462 ┆ 1

In [6]:
scheme_mule = (
    acct_add
    .join(labels, on="account_id", how="left")
    .group_by("scheme_code")
    .agg([
        pl.len().alias("count"),
        pl.col("is_mule").mean().alias("mule_rate")
    ])
    .sort("mule_rate", descending=True)
)
print(scheme_mule)

shape: (7, 3)
┌─────────────┬────────┬───────────┐
│ scheme_code ┆ count  ┆ mule_rate │
│ ---         ┆ ---    ┆ ---       │
│ str         ┆ u32    ┆ f64       │
╞═════════════╪════════╪═══════════╡
│ PMJJBY      ┆ 4804   ┆ 0.033898  │
│ SCSS        ┆ 3203   ┆ 0.031763  │
│ REGULAR     ┆ 115313 ┆ 0.027813  │
│ PMSBY       ┆ 8007   ┆ 0.027661  │
│ PMJDY       ┆ 24022  ┆ 0.027613  │
│ SSA         ┆ 1601   ┆ 0.026652  │
│ APY         ┆ 3203   ┆ 0.022484  │
└─────────────┴────────┴───────────┘


In [7]:
import os
# See actual folder structure
for root, dirs, files in os.walk("../data/transactions"):
    level = root.replace("../data/transactions", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        subindent = " " * 2 * (level + 1)
        # Just show first 3 files per folder
        for f in sorted(files)[:3]:
            print(f"{subindent}{f}")
        if len(files) > 3:
            print(f"{subindent}... ({len(files)} files total)")

transactions/
  batch-1/
    part_0001.parquet
    part_0002.parquet
    part_0003.parquet
    ... (100 files total)
  batch-3/
    part_0201.parquet
    part_0202.parquet
    part_0203.parquet
    ... (100 files total)
  batch-4/
    part_0301.parquet
    part_0302.parquet
    part_0303.parquet
    ... (96 files total)
  batch-2/
    part_0101.parquet
    part_0102.parquet
    part_0103.parquet
    ... (100 files total)


In [8]:
sample = pl.read_parquet("../data/transactions/batch-1/part_0001.parquet")
print(f"Single part shape: {sample.shape}")
print(f"\nSchema:")
print(sample.schema)
print(f"\nChannel distribution:")
print(sample["channel"].value_counts().sort("count", descending=True))
print(f"\nAmount stats:")
print(sample["amount"].describe())

Single part shape: (1000000, 8)

Schema:
Schema({'transaction_id': String, 'account_id': String, 'transaction_timestamp': String, 'mcc_code': Int64, 'channel': String, 'amount': Float64, 'txn_type': String, 'counterparty_id': String})

Channel distribution:
shape: (35, 2)
┌─────────┬────────┐
│ channel ┆ count  │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ UPC     ┆ 379934 │
│ UPD     ┆ 359340 │
│ IPM     ┆ 38212  │
│ P2A     ┆ 27975  │
│ STD     ┆ 23714  │
│ …       ┆ …      │
│ CTC     ┆ 1231   │
│ SID     ┆ 961    │
│ ASD     ┆ 855    │
│ IAD     ┆ 564    │
│ SCW     ┆ 457    │
└─────────┴────────┘

Amount stats:
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ value         │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 1e6           │
│ null_count ┆ 0.0           │
│ mean       ┆ 28484.451339  │
│ std        ┆ 792962.165758 │
│ min        ┆ -6.9809e6     │
│ 25%        ┆ 159.54        │


In [9]:
for root, dirs, files in os.walk("../data/transactions_additional"):
    level = root.replace("../data/transactions_additional", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        subindent = " " * 2 * (level + 1)
        for f in sorted(files)[:3]:
            print(f"{subindent}{f}")
        if len(files) > 3:
            print(f"{subindent}... ({len(files)} files total)")

transactions_additional/
  batch-1/
    part_0001.parquet
    part_0002.parquet
    part_0003.parquet
    ... (100 files total)
  batch-3/
    part_0201.parquet
    part_0202.parquet
    part_0203.parquet
    ... (100 files total)
  batch-4/
    part_0301.parquet
    part_0302.parquet
    part_0303.parquet
    ... (11 files total)
  batch-2/
    part_0101.parquet
    part_0102.parquet
    part_0103.parquet
    ... (100 files total)


In [10]:
sample_add = pl.read_parquet("../data/transactions_additional/batch-1/part_0001.parquet")
print(f"Shape: {sample_add.shape}")
print(f"\nSchema:")
print(sample_add.schema)

total = len(sample_add)
print(f"\nlat null:  {sample_add['latitude'].null_count()/total:.1%}")
print(f"lon null:  {sample_add['longitude'].null_count()/total:.1%}")
print(f"ip null:   {sample_add['ip_address'].null_count()/total:.1%}")
print(f"\nIP sample:")
print(sample_add["ip_address"].drop_nulls().head(5))
print(f"\nPart transaction types:")
print(sample_add["part_transaction_type"].value_counts())

Shape: (1400000, 9)

Schema:
Schema({'transaction_id': String, 'mnemonic_code': String, 'latitude': Float64, 'longitude': Float64, 'ip_address': String, 'balance_after_transaction': Float64, 'part_transaction_type': String, 'atm_deposit_channel_code': String, 'transaction_sub_type': String})

lat null:  73.8%
lon null:  73.8%
ip null:   65.4%

IP sample:
shape: (5,)
Series: 'ip_address' [str]
[
	"122.175.102.100"
	"103.92.190.204"
	"122.124.163.67"
	"103.241.164.38"
	"157.166.43.104"
]

Part transaction types:
shape: (4, 2)
┌───────────────────────┬─────────┐
│ part_transaction_type ┆ count   │
│ ---                   ┆ ---     │
│ str                   ┆ u32     │
╞═══════════════════════╪═════════╡
│ IC                    ┆ 1779    │
│ IP                    ┆ 16514   │
│ CI                    ┆ 1334052 │
│ BI                    ┆ 47655   │
└───────────────────────┴─────────┘


In [ ]:
from glob import glob

print("Starting full transaction scan — this will take 20-40 minutes...")
print("Do not close the notebook. You can minimize it and go to sleep.")

txn_parts = sorted(glob("../data/transactions/batch-*/part_*.parquet"))
print(f"Total parts to scan: {len(txn_parts)}")

txn_lazy = pl.scan_parquet(txn_parts)

basic_txn_features = (
    txn_lazy
    .with_columns([
        pl.col("transaction_timestamp").str.to_datetime(
            format="%Y-%m-%d %H:%M:%S", strict=False
        ).alias("ts")
    ])
    .group_by("account_id")
    .agg([
        pl.len().alias("total_txn_count"),
        pl.col("amount").mean().alias("avg_amount"),
        pl.col("amount").std().alias("std_amount"),
        pl.col("amount").sum().alias("total_amount"),
        pl.col("amount").max().alias("max_amount"),
        pl.col("amount").min().alias("min_amount"),
        (pl.col("amount") < 0).sum().alias("reversal_count"),
        (pl.col("txn_type") == "C").sum().alias("credit_count"),
        (pl.col("txn_type") == "D").sum().alias("debit_count"),
        pl.col("channel").n_unique().alias("unique_channels"),
        pl.col("counterparty_id").n_unique().alias("unique_counterparties"),
        (pl.col("channel") == "ATW").sum().alias("atm_count"),
        (pl.col("channel") == "UPC").sum().alias("upi_credit_count"),
        (pl.col("channel") == "UPD").sum().alias("upi_debit_count"),
        (pl.col("channel") == "IPM").sum().alias("imps_count"),
        (pl.col("channel") == "NTD").sum().alias("neft_count"),
        (pl.col("channel") == "STD").sum().alias("standing_instr_count"),
        pl.col("ts").min().alias("first_txn_date"),
        pl.col("ts").max().alias("last_txn_date"),
    ])
    .collect()
)

output_path = Path("../outputs/features_txn_basic.parquet")
basic_txn_features.write_parquet(output_path)
print(f"\nDone. Shape: {basic_txn_features.shape}")
print(f"Saved to {output_path}")
print(basic_txn_features.head(3))

Starting full transaction scan — this will take 20-40 minutes...
Do not close the notebook. You can minimize it and go to sleep.
Total parts to scan: 396
